# 7. Multi-Scale Seed Sweep — SMALL + MEDIUM GPT (12 runs)

Same multi-seed sweep as notebook 6, but for the **small** and **medium** GPT scales, matched to the large `ctx512 / 3000-step` budget so results are directly comparable across scales.

**GPT identity / W&B id:** `tinystories_{size}_{arch}_ctx512_steps3000_seed{SEED}`

- Sizes: `small` (d384/L6/H6), `medium` (d512/L8/H8)
- Seeds: `123`, `456`, `789`
- Architectures: `baseline`, `attnres`
- Configs: `configs/small_ctx512_3000.yaml`, `configs/medium_ctx512_3000.yaml`

Run setup cells 1–3 first, then run cells 4–15 in order. Each run cell skips automatically if outputs already exist.

## 1. Sync repo and install deps

In [12]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-next-scale.git'


def is_repo_root(path: Path) -> bool:
    return (path / 'src' / 'training' / 'train.py').exists() and (path / 'requirements.txt').exists()


def discover_repo() -> Path | None:
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AttnResGPT-next-scale')]
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate.resolve()
    for root in [Path('/content'), Path('/content/drive/MyDrive')]:
        if not root.exists():
            continue
        for path in root.rglob('AttnResGPT-next-scale'):
            if is_repo_root(path):
                return path.resolve()
    return None


REPO_DIR = discover_repo()
if REPO_DIR is None:
    target = Path('/content/AttnResGPT-next-scale')
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    REPO_DIR = target.resolve()

os.chdir(REPO_DIR)
print(f'Using repo at: {REPO_DIR}')
subprocess.run(['git', 'fetch', '--quiet'], check=False)
subprocess.run(['git', 'pull', '--ff-only'], check=False)
subprocess.run(['git', 'log', '-1', '--oneline'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

Using repo at: C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale


CompletedProcess(args=['c:\\Users\\atin5\\Desktop\\Programming Projects\\voice_cloning\\.conda\\python.exe', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

## 2. W&B + GPU

In [13]:
import os
from pathlib import Path

import torch

# Load WANDB_API_KEY from environment or a local .env (do NOT hardcode secrets).
if not os.environ.get('WANDB_API_KEY'):
    env_path = Path('.env')
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.strip().startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip())

print('WANDB_API_KEY set =', bool(os.environ.get('WANDB_API_KEY')))
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Switch runtime to GPU before launching sweeps.')

WANDB_API_KEY set = True
cuda_available = True
device_name = NVIDIA GeForce RTX 3070 Laptop GPU


## 3. Launch helpers (used by run cells below)

In [19]:
import shutil
import subprocess
import sys
from pathlib import Path

SKIP_EXISTING = True
GPT_WANDB_RUN_SUFFIX = ''  # set to '_rerun1' etc. for a fresh W&B run id
OUTPUT_ROOT = Path('outputs')  # logging.output_root in the configs

CONFIG_BY_SIZE = {
    'small': 'configs/small_ctx512_3000.yaml',
    'medium': 'configs/medium_ctx512_3000.yaml',
}

SIZES = ['small', 'medium']
SEEDS = [123, 456, 789]
ARCHITECTURES = ['baseline', 'attnres']


def repo_root() -> Path:
    """Paths are relative to repo root (cell 1 runs os.chdir(REPO_DIR))."""
    root = Path.cwd().resolve()
    if (root / 'src' / 'training' / 'train.py').exists():
        return root
    fallback = Path('/content/AttnResGPT-next-scale').resolve()
    if (fallback / 'src' / 'training' / 'train.py').exists():
        return fallback
    raise RuntimeError('Run cell 1 first — cannot find repo root.')


def gpt_identity(size: str, architecture: str, seed: int) -> str:
    return f'tinystories_{size}_{architecture}_ctx512_steps3000_seed{seed}'


def gpt_wandb_run_name(size: str, architecture: str, seed: int) -> str:
    return f'{gpt_identity(size, architecture, seed)}{GPT_WANDB_RUN_SUFFIX}'


def gpt_paths(size: str, architecture: str, seed: int, *, root: Path | None = None) -> dict:
    root = root or repo_root()
    identity = gpt_identity(size, architecture, seed)
    run_dir = root / OUTPUT_ROOT / 'runs' / identity
    return {
        'root': root,
        'identity': identity,
        'run_dir': run_dir,
        'checkpoint_dir': root / OUTPUT_ROOT / 'checkpoints' / identity,
        'summary': run_dir / 'run_summary.json',
        'blockers': [
            run_dir / 'train_metrics.jsonl',
            run_dir / 'val_metrics.jsonl',
            run_dir / 'run_summary.json',
        ],
        'global_logs': sorted((root / OUTPUT_ROOT / 'logs').glob(f'{identity}_*.jsonl')),
    }


def reset_gpt_run(size: str, architecture: str, seed: int) -> None:
    """Delete partial/failed run outputs under outputs/ (same dirs train.py uses)."""
    paths = gpt_paths(size, architecture, seed)
    print(f'repo root: {paths["root"]}')
    print(f'resetting: {paths["identity"]}')
    for label in ('run_dir', 'checkpoint_dir'):
        path = paths[label]
        if path.exists():
            shutil.rmtree(path)
            print(f'  removed {label}: {path}')
    for path in paths['global_logs']:
        path.unlink(missing_ok=True)
        print(f'  removed log: {path}')
    remaining = [p for p in paths['blockers'] if p.exists()]
    if remaining:
        raise RuntimeError(f'cleanup incomplete: {remaining}')
    print('  ok — safe to launch')


def launch_gpt(size: str, architecture: str, seed: int) -> None:
    paths = gpt_paths(size, architecture, seed)
    identity = paths['identity']
    root = paths['root']
    config = CONFIG_BY_SIZE[size]
    if SKIP_EXISTING and paths['summary'].exists():
        print(f'SKIP (complete): {identity}')
        return
    if any(p.exists() for p in paths['blockers']):
        print(f'partial outputs for {identity} — clearing before launch')
        reset_gpt_run(size, architecture, seed)
    cmd = [
        sys.executable, '-m', 'src.training.train',
        '--config', config,
        '--model', architecture,
        '--overrides',
        f'experiment.seed={seed}',
        f'logging.wandb.run_name={gpt_wandb_run_name(size, architecture, seed)}',
        f'logging.wandb.tags=[multiseed,{size},ctx512,{architecture},seed{seed}]',
    ]
    print(f'Launching {identity} from {root}')
    print(f'W&B run name: {gpt_wandb_run_name(size, architecture, seed)}')
    subprocess.run(cmd, check=True, cwd=root)

### Run 1/12 — SMALL baseline, seed 123
`tinystories_small_baseline_ctx512_steps3000_seed123`

In [ ]:
launch_gpt('small', 'baseline', 123)

Launching tinystories_small_baseline_ctx512_steps3000_seed123 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_small_baseline_ctx512_steps3000_seed123


### Run 2/12 — SMALL attnres, seed 123
`tinystories_small_attnres_ctx512_steps3000_seed123`

In [ ]:
launch_gpt('small', 'attnres', 123)

### Run 3/12 — SMALL baseline, seed 456
`tinystories_small_baseline_ctx512_steps3000_seed456`

In [ ]:
launch_gpt('small', 'baseline', 456)

### Run 4/12 — SMALL attnres, seed 456 (resume from step 1000)
`tinystories_small_attnres_ctx512_steps3000_seed456`

In [6]:
p = gpt_paths('small', 'attnres', 456)
subprocess.run([
    sys.executable, '-m', 'src.training.train',
    '--config', CONFIG_BY_SIZE['small'], '--model', 'attnres',
    '--overrides',
    'experiment.seed=456',
    f'training.resume_from={(p["checkpoint_dir"] / "step_0001000.pt").as_posix()}',
    f'logging.wandb.run_name={gpt_wandb_run_name("small", "attnres", 456)}',
    'logging.wandb.tags=[multiseed,small,ctx512,attnres,seed456]',
], check=True, cwd=p['root'])

CompletedProcess(args=['c:\\Users\\atin5\\Desktop\\Programming Projects\\voice_cloning\\.conda\\python.exe', '-m', 'src.training.train', '--config', 'configs/small_ctx512_3000.yaml', '--model', 'attnres', '--overrides', 'experiment.seed=456', 'training.resume_from=C:/Users/atin5/Desktop/Programming Projects/AttnResGPT-next-scale/outputs/checkpoints/tinystories_small_attnres_ctx512_steps3000_seed456/step_0001000.pt', 'logging.wandb.run_name=tinystories_small_attnres_ctx512_steps3000_seed456', 'logging.wandb.tags=[multiseed,small,ctx512,attnres,seed456]'], returncode=0)

### Run 5/12 — SMALL baseline, seed 789
`tinystories_small_baseline_ctx512_steps3000_seed789`

In [7]:
launch_gpt('small', 'baseline', 789)

Launching tinystories_small_baseline_ctx512_steps3000_seed789 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_small_baseline_ctx512_steps3000_seed789


### Run 6/12 — SMALL attnres, seed 789
`tinystories_small_attnres_ctx512_steps3000_seed789`

In [8]:
launch_gpt('small', 'attnres', 789)

Launching tinystories_small_attnres_ctx512_steps3000_seed789 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_small_attnres_ctx512_steps3000_seed789


### Run 7/12 — MEDIUM baseline, seed 123 (resume from step 1000)
`tinystories_medium_baseline_ctx512_steps3000_seed123`

In [27]:
launch_gpt('medium', 'baseline', 123)

partial outputs for tinystories_medium_baseline_ctx512_steps3000_seed123 — clearing before launch
repo root: C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
resetting: tinystories_medium_baseline_ctx512_steps3000_seed123
  removed run_dir: C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale\outputs\runs\tinystories_medium_baseline_ctx512_steps3000_seed123
  removed checkpoint_dir: C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale\outputs\checkpoints\tinystories_medium_baseline_ctx512_steps3000_seed123
  removed log: C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale\outputs\logs\tinystories_medium_baseline_ctx512_steps3000_seed123_train.jsonl
  removed log: C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale\outputs\logs\tinystories_medium_baseline_ctx512_steps3000_seed123_val.jsonl
  ok — safe to launch
Launching tinystories_medium_baseline_ctx512_steps3000_seed123 from C:\Users\atin5\Desktop\Programming Proj

### Run 8/12 — MEDIUM attnres, seed 123
`tinystories_medium_attnres_ctx512_steps3000_seed123`

In [21]:
launch_gpt('medium', 'attnres', 123)

Launching tinystories_medium_attnres_ctx512_steps3000_seed123 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_medium_attnres_ctx512_steps3000_seed123


### Run 9/12 — MEDIUM baseline, seed 456
`tinystories_medium_baseline_ctx512_steps3000_seed456`

In [22]:
launch_gpt('medium', 'baseline', 456)

Launching tinystories_medium_baseline_ctx512_steps3000_seed456 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_medium_baseline_ctx512_steps3000_seed456


### Run 10/12 — MEDIUM attnres, seed 456
`tinystories_medium_attnres_ctx512_steps3000_seed456`

In [23]:
launch_gpt('medium', 'attnres', 456)

Launching tinystories_medium_attnres_ctx512_steps3000_seed456 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_medium_attnres_ctx512_steps3000_seed456


### Run 11/12 — MEDIUM baseline, seed 789
`tinystories_medium_baseline_ctx512_steps3000_seed789`

In [24]:
launch_gpt('medium', 'baseline', 789)

Launching tinystories_medium_baseline_ctx512_steps3000_seed789 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_medium_baseline_ctx512_steps3000_seed789


### Run 12/12 — MEDIUM attnres, seed 789
`tinystories_medium_attnres_ctx512_steps3000_seed789`

In [25]:
launch_gpt('medium', 'attnres', 789)

Launching tinystories_medium_attnres_ctx512_steps3000_seed789 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_medium_attnres_ctx512_steps3000_seed789


## Optional: run the whole sweep in one loop
Instead of the per-run cells above, this launches all 12 runs sequentially (small → medium, seeds 123/456/789, baseline + attnres). Completed runs are skipped automatically.

In [4]:
for size in SIZES:
    for seed in SEEDS:
        for architecture in ARCHITECTURES:
            launch_gpt(size, architecture, seed)

Launching tinystories_small_baseline_ctx512_steps3000_seed123 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_small_baseline_ctx512_steps3000_seed123
Launching tinystories_small_attnres_ctx512_steps3000_seed123 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_small_attnres_ctx512_steps3000_seed123
Launching tinystories_small_baseline_ctx512_steps3000_seed456 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_small_baseline_ctx512_steps3000_seed456
Launching tinystories_small_attnres_ctx512_steps3000_seed456 from C:\Users\atin5\Desktop\Programming Projects\AttnResGPT-next-scale
W&B run name: tinystories_small_attnres_ctx512_steps3000_seed456


CalledProcessError: Command '['c:\\Users\\atin5\\Desktop\\Programming Projects\\voice_cloning\\.conda\\python.exe', '-m', 'src.training.train', '--config', 'configs/small_ctx512_3000.yaml', '--model', 'attnres', '--overrides', 'experiment.seed=456', 'logging.wandb.run_name=tinystories_small_attnres_ctx512_steps3000_seed456', 'logging.wandb.tags=[multiseed,small,ctx512,attnres,seed456]']' returned non-zero exit status 1.

## Post-sweep sanity check

In [28]:
import json
from pathlib import Path

print('=== GPT summaries (small + medium, ctx512/3000) ===')
for size in SIZES:
    for architecture in ARCHITECTURES:
        for seed in SEEDS:
            identity = gpt_identity(size, architecture, seed)
            summary_path = Path('outputs/runs') / identity / 'run_summary.json'
            if not summary_path.exists():
                print(f'MISSING {identity}')
                continue
            summary = json.loads(summary_path.read_text())
            print(
                f"{identity}: val_loss={summary.get('val_loss'):.4f} "
                f"ppl={summary.get('perplexity'):.2f}"
            )

=== GPT summaries (small + medium, ctx512/3000) ===
tinystories_small_baseline_ctx512_steps3000_seed123: val_loss=2.4800 ppl=11.94
tinystories_small_baseline_ctx512_steps3000_seed456: val_loss=2.4826 ppl=11.97
tinystories_small_baseline_ctx512_steps3000_seed789: val_loss=2.4738 ppl=11.87
tinystories_small_attnres_ctx512_steps3000_seed123: val_loss=2.3937 ppl=10.95
tinystories_small_attnres_ctx512_steps3000_seed456: val_loss=2.4011 ppl=11.04
tinystories_small_attnres_ctx512_steps3000_seed789: val_loss=2.4044 ppl=11.07
tinystories_medium_baseline_ctx512_steps3000_seed123: val_loss=2.2819 ppl=9.80
tinystories_medium_baseline_ctx512_steps3000_seed456: val_loss=2.2706 ppl=9.69
tinystories_medium_baseline_ctx512_steps3000_seed789: val_loss=2.2800 ppl=9.78
tinystories_medium_attnres_ctx512_steps3000_seed123: val_loss=2.1360 ppl=8.47
tinystories_medium_attnres_ctx512_steps3000_seed456: val_loss=2.1260 ppl=8.38
tinystories_medium_attnres_ctx512_steps3000_seed789: val_loss=2.1436 ppl=8.53
